# Parihaka facies: 5-seed ensemble, uncertainty, and distance analysis
Set **Runtime > Change runtime type > T4 GPU** first. Run the code cells top to bottom.
Total time: about 10 min setup + about 45 min training + a few minutes of evaluation.

In [ ]:
!nvidia-smi -L

## 1. Code and dependencies

In [ ]:
!test -d Wildcat-Neural-Network || git clone https://github.com/kten314/Wildcat-Neural-Network.git
%cd Wildcat-Neural-Network
!git pull -q
!pip install -q zarr segmentation-models-pytorch

## 2. Data (sizes should be about 1.6 GB and 6.9 MB)

In [ ]:
!mkdir -p data/raw/parihaka
!curl -fL -o data/raw/parihaka/parihaka_labels_train.npz "https://data.mendeley.com/public-files/datasets/gnvyh3msrj/files/0a0ac0af-10ab-47c3-83ac-f38fb0090d4a/file_downloaded"
!curl -fL -o data/raw/parihaka/parihaka_data_train.npz "https://data.mendeley.com/public-files/datasets/gnvyh3msrj/files/2777ae64-68e3-4cb0-aae4-588b30b7009c/file_downloaded"
!ls -lh data/raw/parihaka
!file data/raw/parihaka/*.npz

## 3. Convert to Zarr (about 8 minutes)

In [ ]:
!python src/convert_parihaka.py

## 4. Quick check of the evaluation pipeline (about 5 minutes)
Trains one tiny model and runs the full evaluation on it. The scores mean nothing here; this only
confirms the pipeline runs end to end before you spend 45 minutes training. If it fails, stop and send the error.

In [ ]:
!python src/train_unet.py --epochs 1 --steps-per-epoch 10 --tag pipecheck --figs 0
!python src/ensemble_eval.py --tag pipecheck --figs 1

## 5. Train five ensemble members (about 40-45 minutes)
Seeds 0 to 4, 30 epochs each. Skips any seed whose checkpoint already exists, so you can re-run this cell
after an interruption (as long as the session was not reset). Keep this tab open while it runs.

In [ ]:
import os
for s in range(5):
    if os.path.exists(f"checkpoints/ens_seed{s}.pt"):
        print(f"seed {s} already trained, skipping")
    else:
        print(f"===== training seed {s} =====")
        get_ipython().system(f"python src/train_unet.py --epochs 30 --steps-per-epoch 100 --batch 16 --tag ens --seed {s} --figs 0")

## 6. Evaluate the ensemble

In [ ]:
!python src/ensemble_eval.py --tag ens --figs 3

## 7. Look at the figures

In [ ]:
from IPython.display import Image, display
for name in ["ensemble_ens_distance", "ensemble_ens_uncertainty", "ensemble_ens_gallery"]:
    display(Image(f"results/figures/{name}.png"))

## 8. Download results (JSON and figures only, a few MB)

In [ ]:
!zip -qr ensemble_results.zip results/ensemble_ens.json results/figures/ensemble_ens_distance.png results/figures/ensemble_ens_uncertainty.png results/figures/ensemble_ens_gallery.png results/ens_seed*.json
from google.colab import files
files.download("ensemble_results.zip")